# MemAgent with E2B Sandbox — Code Execution in the Cloud

This notebook demonstrates how to use **MemAgent** with the **E2B** sandbox provider to give your AI agent the ability to **generate code, execute it in an isolated cloud environment, and use the output** to answer questions.

## What is E2B?

E2B is a cloud-based code execution sandbox built specifically for AI agents. It uses **Firecracker microVMs** to provide:
- Secure, isolated execution (~150ms cold start)
- Full Linux OS access inside the sandbox
- File system read/write operations
- Internet access from inside the sandbox

## What You'll Learn

1. How to set up MemAgent with E2B sandbox
2. How the agent generates and executes code autonomously
3. How to use sandbox file operations
4. How sandbox tools integrate with MemAgent's memory system

---
## Step 1: Install Dependencies

Install MemoRizz with E2B sandbox support and the OpenAI SDK for LLM access.

In [ ]:
# Install memorizz with E2B sandbox support
%pip install -qU memorizz
%pip install -qU e2b-code-interpreter
%pip install -qU openai

print("All packages installed successfully!")

---
## Step 2: Configure API Keys

You'll need two API keys:
- **E2B API Key** — Get one free at [e2b.dev](https://e2b.dev) ($100 free credits)
- **OpenAI API Key** — Get one at [platform.openai.com](https://platform.openai.com)

In [ ]:
import os
import getpass

def set_env_securely(var_name, prompt):
    """Securely prompt for and set an environment variable."""
    value = getpass.getpass(prompt)
    os.environ[var_name] = value

# Set your API keys (these will prompt securely)
set_env_securely("E2B_API_KEY", "Enter your E2B API key: ")
set_env_securely("OPENAI_API_KEY", "Enter your OpenAI API key: ")

---
## Step 3: Create a MemAgent with E2B Sandbox

Adding sandbox capabilities to MemAgent is a single parameter: `sandbox_provider="e2b"`. This automatically registers three tools the LLM can call:

| Tool | Description |
|------|-------------|
| `execute_code(code, language)` | Run code in the sandbox |
| `sandbox_write_file(path, content)` | Write files in the sandbox |
| `sandbox_read_file(path)` | Read files from the sandbox |

In [ ]:
import logging

# Configure logging so we can see tool calls
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    force=True
)

In [ ]:
from memorizz.memagent.core import MemAgent
from memorizz.llms.llm_factory import create_llm_provider

# Create the LLM provider
llm = create_llm_provider({
    "provider": "openai",
    "model": "gpt-4o",
    "api_key": os.getenv("OPENAI_API_KEY"),
})

# Create MemAgent with E2B sandbox
agent = MemAgent(
    model=llm,
    sandbox_provider="e2b",  # This is all it takes!
    instruction=(
        "You are a helpful data scientist assistant. "
        "When users ask questions that require computation, "
        "write Python code and execute it using the execute_code tool. "
        "Always show your work by explaining what the code does."
    ),
)

print(f"Agent ID: {agent.agent_id}")
print(f"Sandbox provider: {agent.get_sandbox_provider_name()}")
print(f"Has sandbox: {agent.has_sandbox()}")
print(f"Tools registered: {agent.tool_manager.list_tools()}")

---
## Step 4: Let the Agent Generate and Execute Code

Now let's ask the agent a question that requires computation. The agent will:
1. Understand it needs to write code
2. Generate Python code
3. Call the `execute_code` tool to run it in the E2B sandbox
4. Read the output and formulate a response

### Example 1: Mathematical Computation

In [ ]:
response = agent.run(
    "What is the 50th Fibonacci number? Show your work by computing it."
)
print(f"\nAgent: {response}")

### Example 2: Data Analysis

The agent can perform more complex data analysis tasks. The sandbox has Python's standard library available, plus common packages like `numpy`.

In [ ]:
response = agent.run(
    "Generate a list of 100 random numbers between 1 and 1000, "
    "then calculate the mean, median, standard deviation, "
    "and find the top 5 largest numbers."
)
print(f"\nAgent: {response}")

### Example 3: String Processing and Algorithms

The agent can also write algorithmic code to solve problems.

In [ ]:
response = agent.run(
    "Write a Python function that checks if a string is a palindrome, "
    "then test it with these words: 'racecar', 'hello', 'madam', 'python', 'level'. "
    "Show the results."
)
print(f"\nAgent: {response}")

---
## Step 5: Direct Code Execution

You can also call `execute_code` directly from Python without going through the LLM. This is useful for testing or programmatic code execution.

In [ ]:
import json

# Execute code directly through the agent
result_json = agent.execute_code("""
import sys
print(f"Python version: {sys.version}")
print(f"Platform: {sys.platform}")

# Simple computation
numbers = [i**2 for i in range(1, 11)]
print(f"First 10 perfect squares: {numbers}")
print(f"Sum: {sum(numbers)}")
""")

result = json.loads(result_json)
print("Execution result:")
print(f"  Success: {result['success']}")
print(f"  Output:")
for line in result['stdout']:
    print(f"    {line}")

---
## Step 6: Iterative Problem Solving

One of the most powerful aspects of sandbox execution is that the agent can **iterate**. If code fails, the agent sees the error, fixes it, and tries again — all within the same `run()` call.

In [ ]:
response = agent.run(
    "Write a function to find all prime numbers up to 200 using the Sieve of Eratosthenes. "
    "Execute it, print the primes, count them, and tell me how many there are."
)
print(f"\nAgent: {response}")

---
## Step 7: Runtime Provider Management

You can swap or disable the sandbox provider at runtime using `with_sandbox_provider()`.

In [ ]:
# Check current state
print(f"Has sandbox: {agent.has_sandbox()}")
print(f"Provider: {agent.get_sandbox_provider_name()}")

# Disable sandbox
agent.with_sandbox_provider(None)
print(f"\nAfter disabling:")
print(f"Has sandbox: {agent.has_sandbox()}")

# Re-enable with E2B
agent.with_sandbox_provider("e2b")
print(f"\nAfter re-enabling:")
print(f"Has sandbox: {agent.has_sandbox()}")
print(f"Provider: {agent.get_sandbox_provider_name()}")

---
## Key Takeaways

1. **One parameter** — Adding `sandbox_provider="e2b"` to MemAgent gives it full code execution capabilities.
2. **Stateless execution** — Each `execute_code` call runs in a fresh sandbox. Variables don't persist between calls.
3. **Auto-registered tools** — Three tools (`execute_code`, `sandbox_write_file`, `sandbox_read_file`) are registered automatically.
4. **Iterative reasoning** — The agent can generate code, see errors, and fix them within a single `run()` call.
5. **Secure isolation** — Code runs in Firecracker microVMs, completely isolated from your machine.

## Next Steps

- Try the **Daytona sandbox** notebook for GPU support and unlimited runtime
- Try the **GraalPy sandbox** notebook for local execution without cloud dependencies
- Combine sandbox with **memory providers** for persistent agent state